## Setup and Data Loading

In [25]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from csdid.att_gt import ATTgt as att_gt
from csdid.aggte_fnc.aggte import aggte as aggregate
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [26]:
# Load the dataset
url = "https://raw.githubusercontent.com/LOST-STATS/LOST-STATS.github.io/master/Model_Estimation/Data/Event_Study_DiD/bacon_example.csv"
df = pd.read_csv(url)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

Dataset loaded successfully!
Shape: (1617, 10)


,stfips,year,_nfd,post,asmrs,pcinc,asmrh,cases,weight,copop
0,1,1964,1971.0,0,35.639885,12406.178537,5.007341,0.012312,1715156.0,1.715156e+06
1,1,1965,1971.0,0,41.543755,13070.206738,4.425367,0.010419,1715156.0,1.725186e+06
2,1,1966,1971.0,0,34.252335,13526.663217,4.874819,0.009900,1715156.0,1.735219e+06
3,1,1967,1971.0,0,34.465023,13918.189823,5.362014,0.009975,1715156.0,1.745250e+06
4,1,1968,1971.0,0,40.440105,14684.808682,4.643759,0.012401,1715156.0,1.755283e+06


In [27]:
# Prepare data for CSDiD
# CSDiD requires:
# - yname: outcome variable
# - tname: time variable
# - idname: unit identifier
# - gname: group variable (first treatment period, 0 for never-treated)

# The _nfd column contains the first treatment period (0 for never-treated)
# This is exactly what CSDiD needs for the gname

print("Treatment groups (first treatment period):")
print(df['_nfd'].value_counts().sort_index())
print(f"\nNote: _nfd = 0 indicates never-treated units")

Treatment groups (first treatment period):
_nfd
1969.0     66
1970.0     66
1971.0    231
1972.0     99
1973.0    330
1974.0     99
1975.0     66
1976.0     33
1977.0     99
1980.0     33
1984.0     33
1985.0     33
Name: count, dtype: int64

Note: _nfd = 0 indicates never-treated units


In [28]:
# Create a clean copy for CSDiD
df_csdid = df.copy()

# Ensure the data is properly formatted
print("\nData summary for CSDiD:")
print(f"  Outcome variable (asmrs): {df_csdid['asmrs'].describe()['mean']:.2f} (mean)")
print(f"  Unit identifier (stfips): {df_csdid['stfips'].nunique()} unique units")
print(f"  Time variable (year): {df_csdid['year'].min()} to {df_csdid['year'].max()}")
print(f"  Group variable (_nfd): {df_csdid['_nfd'].nunique()} treatment cohorts")


Data summary for CSDiD:
  Outcome variable (asmrs): 52.17 (mean)
  Unit identifier (stfips): 49 unique units
  Time variable (year): 1964 to 1996
  Group variable (_nfd): 12 treatment cohorts


---

## Part a) CSDiD Estimation: ATT(g,t)

The Callaway and Sant'Anna (2021) estimator computes group-time average treatment effects:

$$ATT(g,t) = E[Y_t(1) - Y_t(0) | G_g = 1]$$

Where:
- $g$ = treatment cohort (first treatment period)
- $t$ = calendar time period
- $G_g = 1$ indicates units first treated in period $g$

In [29]:
# Estimate ATT(g,t) using CSDiD
# We include pcinc, asmrh, and cases as control variables

att_gt_results = att_gt(
    data=df_csdid,
    yname='asmrs',           # Outcome variable
    tname='year',            # Time variable
    idname='stfips',         # Unit identifier
    gname='_nfd',            # Group (first treatment period)
    xformla='~pcinc+asmrh+cases',  # Control variables formula
    control_group='nevertreated'  # Use never-treated as control group
)

print("CSDiD estimation completed!")
print(f"\nNumber of ATT(g,t) estimates: {len(att_gt_results['att'])}")

dropped, 429, rows from original data due to missing data
CSDiD estimation completed!


TypeError: 'ATTgt' object is not subscriptable

In [ ]:
# Create a clean table of ATT(g,t) results
att_gt_df = pd.DataFrame({
    'Group (g)': att_gt_results['group'],
    'Time (t)': att_gt_results['t'],
    'ATT(g,t)': att_gt_results['att'],
    'Std. Error': att_gt_results['se'],
    'CI Lower': att_gt_results['att'] - 1.96 * att_gt_results['se'],
    'CI Upper': att_gt_results['att'] + 1.96 * att_gt_results['se']
})

# Add significance indicators
att_gt_df['Significant'] = np.where(
    (att_gt_df['CI Lower'] > 0) | (att_gt_df['CI Upper'] < 0),
    '***',
    ''
)

print("="*90)
print("ATT(g,t) ESTIMATES - CALLAWAY-SANT'ANNA DIFFERENCE-IN-DIFFERENCES")
print("="*90)
print(att_gt_df.round(4).to_string(index=False))

In [ ]:
# Create a pivot table for better visualization
att_pivot = att_gt_df.pivot_table(
    index='Group (g)',
    columns='Time (t)',
    values='ATT(g,t)',
    aggfunc='first'
).round(4)

print("\n" + "="*90)
print("ATT(g,t) PIVOT TABLE")
print("Rows: Treatment Cohort (g) | Columns: Calendar Time (t)")
print("="*90)
print(att_pivot)

# Save to CSV
att_gt_df.to_csv('../output/att_gt_results.csv', index=False)
print("\nResults saved to: ../output/att_gt_results.csv")

In [ ]:
# Visualize ATT(g,t) as a heatmap
fig, ax = plt.subplots(figsize=(16, 8))

# Create heatmap
sns.heatmap(
    att_pivot,
    cmap='RdBu_r',
    center=0,
    annot=True,
    fmt='.1f',
    cbar_kws={'label': 'ATT(g,t)'},
    ax=ax,
    linewidths=0.5
)

ax.set_title('ATT(g,t) Heatmap: Group-Time Treatment Effects\n(Callaway-Sant\'Anna DiD)', fontsize=16)
ax.set_xlabel('Calendar Time (t)', fontsize=14)
ax.set_ylabel('Treatment Cohort (g)', fontsize=14)

plt.tight_layout()
plt.savefig('../output/att_gt_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../output/att_gt_heatmap.png")

---

## Part b) Aggregations

CSDiD provides three main aggregation methods to summarize the group-time effects.

### b.1) Aggregation by Group

Aggregates ATT(g,t) across all post-treatment periods for each treatment cohort:

$$ATT_{group}(g) = \frac{1}{|\{t: t \geq g\}|} \sum_{t \geq g} ATT(g,t)$$

In [ ]:
# Aggregation by Group
agg_group = aggregate(att_gt_results, type='group')

# Create summary table
agg_group_df = pd.DataFrame({
    'Group (g)': agg_group['egt'],
    'ATT': agg_group['att.egt'],
    'Std. Error': agg_group['se.egt'],
    'CI Lower': agg_group['att.egt'] - 1.96 * agg_group['se.egt'],
    'CI Upper': agg_group['att.egt'] + 1.96 * agg_group['se.egt']
})

print("="*80)
print("AGGREGATION BY GROUP")
print("Average treatment effect for each treatment cohort (across all post-treatment periods)")
print("="*80)
print(agg_group_df.round(4).to_string(index=False))

# Overall ATT
print(f"\n*** Overall ATT: {agg_group['overall.att']:.4f} (SE: {agg_group['overall.se']:.4f}) ***")

In [ ]:
# Plot aggregation by group
fig, ax = plt.subplots(figsize=(10, 6))

ax.errorbar(
    agg_group_df['Group (g)'],
    agg_group_df['ATT'],
    yerr=1.96 * agg_group_df['Std. Error'],
    fmt='o',
    markersize=10,
    capsize=5,
    capthick=2,
    color='darkgreen',
    ecolor='green',
    label='ATT by Group ± 95% CI'
)

ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.axhline(y=agg_group['overall.att'], color='red', linestyle='--', linewidth=2, 
           label=f"Overall ATT = {agg_group['overall.att']:.2f}")

ax.set_xlabel('Treatment Cohort (First Treatment Year)', fontsize=14)
ax.set_ylabel('Average Treatment Effect', fontsize=14)
ax.set_title('CSDiD: Aggregation by Group\n(Average effect for each treatment cohort)', fontsize=16)
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('../output/csdid_aggregation_group.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../output/csdid_aggregation_group.png")

### b.2) Aggregation by Period (Calendar Time)

Aggregates ATT(g,t) across all treated groups for each calendar period:

$$ATT_{calendar}(t) = \sum_{g \leq t} w_g \cdot ATT(g,t)$$

Where $w_g$ is the share of group $g$ among all treated units at time $t$.

In [ ]:
# Aggregation by Calendar Time
agg_time = aggregate(att_gt_results, type='calendar')

# Create summary table
agg_time_df = pd.DataFrame({
    'Time (t)': agg_time['egt'],
    'ATT': agg_time['att.egt'],
    'Std. Error': agg_time['se.egt'],
    'CI Lower': agg_time['att.egt'] - 1.96 * agg_time['se.egt'],
    'CI Upper': agg_time['att.egt'] + 1.96 * agg_time['se.egt']
})

print("="*80)
print("AGGREGATION BY CALENDAR TIME (PERIOD)")
print("Average treatment effect for each calendar period (across all treated groups)")
print("="*80)
print(agg_time_df.round(4).to_string(index=False))

# Overall ATT
print(f"\n*** Overall ATT: {agg_time['overall.att']:.4f} (SE: {agg_time['overall.se']:.4f}) ***")

In [ ]:
# Plot aggregation by time
fig, ax = plt.subplots(figsize=(12, 6))

ax.errorbar(
    agg_time_df['Time (t)'],
    agg_time_df['ATT'],
    yerr=1.96 * agg_time_df['Std. Error'],
    fmt='s-',
    markersize=8,
    capsize=4,
    capthick=2,
    color='darkblue',
    ecolor='steelblue',
    label='ATT by Calendar Time ± 95% CI'
)

ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.axhline(y=agg_time['overall.att'], color='red', linestyle='--', linewidth=2,
           label=f"Overall ATT = {agg_time['overall.att']:.2f}")

ax.set_xlabel('Calendar Year', fontsize=14)
ax.set_ylabel('Average Treatment Effect', fontsize=14)
ax.set_title('CSDiD: Aggregation by Calendar Time\n(Average effect in each year across all treated units)', fontsize=16)
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('../output/csdid_aggregation_calendar.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../output/csdid_aggregation_calendar.png")

### b.3) Aggregation by Event-Time (Dynamic Effects)

Aggregates ATT(g,t) across all groups for each event-time (relative time since treatment):

$$ATT_{event}(e) = \sum_{g} w_g \cdot ATT(g, g+e)$$

Where $e = t - g$ is the event time (years since treatment).

In [ ]:
# Aggregation by Event Time (Dynamic Effects)
agg_event = aggregate(att_gt_results, type='dynamic')

# Create summary table
agg_event_df = pd.DataFrame({
    'Event Time (e)': agg_event['egt'],
    'ATT': agg_event['att.egt'],
    'Std. Error': agg_event['se.egt'],
    'CI Lower': agg_event['att.egt'] - 1.96 * agg_event['se.egt'],
    'CI Upper': agg_event['att.egt'] + 1.96 * agg_event['se.egt']
}).sort_values('Event Time (e)').reset_index(drop=True)

print("="*80)
print("AGGREGATION BY EVENT-TIME (DYNAMIC EFFECTS)")
print("Average treatment effect at each relative time since treatment")
print("="*80)
print(agg_event_df.round(4).to_string(index=False))

# Overall ATT
print(f"\n*** Overall ATT: {agg_event['overall.att']:.4f} (SE: {agg_event['overall.se']:.4f}) ***")

# Save event-time aggregation for comparison
agg_event_df.to_csv('../output/csdid_event_time.csv', index=False)
print("\nEvent-time results saved to: ../output/csdid_event_time.csv")

In [ ]:
# Plot dynamic effects (event-time aggregation)
fig, ax = plt.subplots(figsize=(14, 8))

# Plot coefficients with confidence intervals
ax.errorbar(
    agg_event_df['Event Time (e)'],
    agg_event_df['ATT'],
    yerr=1.96 * agg_event_df['Std. Error'],
    fmt='o',
    markersize=8,
    capsize=4,
    capthick=2,
    color='darkred',
    ecolor='indianred',
    label='CSDiD Event-Time ATT ± 95% CI'
)

# Add connecting line
ax.plot(agg_event_df['Event Time (e)'], agg_event_df['ATT'], 
        'o-', color='darkred', alpha=0.7, markersize=8)

# Reference lines
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.axvline(x=-0.5, color='blue', linestyle='--', linewidth=2, label='Treatment (e=0)')

# Shading
ax.axvspan(agg_event_df['Event Time (e)'].min() - 0.5, -0.5, alpha=0.1, color='blue', label='Pre-treatment')
ax.axvspan(-0.5, agg_event_df['Event Time (e)'].max() + 0.5, alpha=0.1, color='orange', label='Post-treatment')

ax.set_xlabel('Event Time (Years Since Treatment)', fontsize=14)
ax.set_ylabel('Average Treatment Effect', fontsize=14)
ax.set_title('CSDiD: Dynamic Effects (Event-Time Aggregation)\n(Callaway-Sant\'Anna Difference-in-Differences)', fontsize=16)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/csdid_event_study.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved to: ../output/csdid_event_study.png")

---

## Part c) Explanation of Aggregations

### c.1) Aggregation by Group

**Meaning:** The group-specific ATT gives the average treatment effect for units in a particular treatment cohort, averaged across all post-treatment periods they experience.

**Interpretation:** This tells us whether the treatment had different effects on different cohorts. For example, early adopters might experience different effects than late adopters due to:
- Different baseline characteristics
- Changes in the policy implementation over time
- Time-varying heterogeneity in treatment effects

**Use case:** Understanding heterogeneity across treatment cohorts.

---

### c.2) Aggregation by Period (Calendar Time)

**Meaning:** The calendar-time ATT gives the average treatment effect in a specific calendar year, weighted by the proportion of each treated cohort.

**Interpretation:** This shows how the treatment effect evolved over calendar time. The composition of treated units changes as more cohorts become treated.

**Use case:** Policy evaluation that requires understanding effects at specific points in time.

---

### c.3) Aggregation by Event-Time

**Meaning:** The event-time ATT gives the average treatment effect at specific periods relative to treatment (e.g., 1 year after, 2 years after), pooled across all treatment cohorts.

**Interpretation:** This is analogous to a traditional event-study plot, showing:
- Pre-trends (event time < 0): Should be near zero if parallel trends holds
- Impact effects (event time = 0): Immediate treatment effect
- Dynamic effects (event time > 0): How effects evolve after treatment

**Use case:** Understanding the dynamics of treatment effects over time since treatment.

---

### c.4) Which aggregation is most comparable to TWFE Event-Study coefficients?

**Answer: Event-Time Aggregation**

The **event-time aggregation** from CSDiD is most comparable to the TWFE event-study coefficients because:

1. Both measure effects at specific periods relative to treatment (event time)
2. Both show the dynamic pattern of treatment effects
3. The reference period (usually t = -1) is normalized to zero in both

**Key difference:** CSDiD uses only clean comparisons (never-treated or not-yet-treated as controls), while TWFE may use already-treated units as implicit controls, leading to potential bias.

**Implication:** Differences between CSDiD event-time ATT and TWFE event-study coefficients can reveal the extent of bias in the traditional TWFE approach.

---

## Part d) Comparison: CSDiD vs TWFE Event-Study

In [ ]:
# Load the TWFE event-study results from Question 1
try:
    twfe_es_df = pd.read_csv('../output/event_study_coefficients.csv')
    print("TWFE Event-Study results loaded successfully!")
    print(twfe_es_df.head())
except FileNotFoundError:
    print("Note: TWFE results not found. Please run Q1 notebook first.")
    # Create placeholder for demonstration
    twfe_es_df = None

In [ ]:
# Create comparison table
if twfe_es_df is not None:
    # Prepare CSDiD data
    csdid_compare = agg_event_df[['Event Time (e)', 'ATT', 'Std. Error']].copy()
    csdid_compare.columns = ['Event Time', 'CSDiD ATT', 'CSDiD SE']
    
    # Prepare TWFE data
    twfe_compare = twfe_es_df[['event_time', 'coefficient', 'std_error']].copy()
    twfe_compare.columns = ['Event Time', 'TWFE Coef', 'TWFE SE']
    
    # Merge
    comparison_df = pd.merge(
        csdid_compare,
        twfe_compare,
        on='Event Time',
        how='outer'
    ).sort_values('Event Time').reset_index(drop=True)
    
    # Calculate difference
    comparison_df['Difference'] = comparison_df['CSDiD ATT'] - comparison_df['TWFE Coef']
    
    print("="*90)
    print("COMPARISON TABLE: CSDiD vs TWFE EVENT-STUDY")
    print("="*90)
    print(comparison_df.round(4).to_string(index=False))
    
    # Save comparison table
    comparison_df.to_csv('../output/comparison_csdid_twfe.csv', index=False)
    print("\nComparison table saved to: ../output/comparison_csdid_twfe.csv")
else:
    print("Cannot create comparison table - TWFE results not available.")

In [ ]:
# Create combined coefficient plot
if twfe_es_df is not None:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Plot CSDiD
    ax.errorbar(
        agg_event_df['Event Time (e)'] - 0.1,  # Slight offset for visibility
        agg_event_df['ATT'],
        yerr=1.96 * agg_event_df['Std. Error'],
        fmt='o',
        markersize=9,
        capsize=4,
        capthick=2,
        color='darkred',
        ecolor='indianred',
        label='CSDiD Event-Time ATT'
    )
    
    # Plot TWFE
    ax.errorbar(
        twfe_es_df['event_time'] + 0.1,  # Slight offset for visibility
        twfe_es_df['coefficient'],
        yerr=1.96 * twfe_es_df['std_error'],
        fmt='s',
        markersize=8,
        capsize=4,
        capthick=2,
        color='navy',
        ecolor='steelblue',
        label='TWFE Event-Study'
    )
    
    # Connect points with lines
    ax.plot(agg_event_df['Event Time (e)'], agg_event_df['ATT'], 
            '-', color='darkred', alpha=0.5)
    ax.plot(twfe_es_df['event_time'], twfe_es_df['coefficient'], 
            '-', color='navy', alpha=0.5)
    
    # Reference lines
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.axvline(x=-0.5, color='green', linestyle='--', linewidth=2, label='Treatment')
    
    # Formatting
    ax.set_xlabel('Event Time (Years Since Treatment)', fontsize=14)
    ax.set_ylabel('Coefficient / ATT Estimate', fontsize=14)
    ax.set_title('Comparison: CSDiD Event-Time ATT vs TWFE Event-Study Coefficients\n(95% Confidence Intervals)', fontsize=16)
    ax.legend(fontsize=12, loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../output/comparison_plot_csdid_twfe.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nFigure saved to: ../output/comparison_plot_csdid_twfe.png")
else:
    print("Cannot create comparison plot - TWFE results not available.")

### Brief Comparison Discussion

**Key Observations:**

1. **Pre-treatment period (event time < 0):**
   - Both methods should show near-zero effects if parallel trends holds
   - Differences in pre-trends can indicate violations of parallel trends assumption
   - CSDiD pre-trends are often closer to zero as it uses cleaner comparisons

2. **Treatment effect magnitude:**
   - TWFE may be biased when there are heterogeneous treatment effects across cohorts
   - CSDiD provides unbiased estimates under weaker assumptions
   - The direction of bias in TWFE depends on the pattern of effect heterogeneity

3. **Precision:**
   - TWFE standard errors may be smaller (but misleading if biased)
   - CSDiD standard errors properly account for estimation uncertainty

4. **Interpretation:**
   - If CSDiD and TWFE estimates are similar → less concern about TWFE bias
   - If estimates differ substantially → TWFE bias is likely present
   - The Bacon decomposition can help understand sources of TWFE bias

**Recommendation:** In settings with staggered treatment and potential heterogeneous effects, CSDiD provides more reliable estimates than TWFE. Always compare both methods as a robustness check.

---

## Summary

### Key Results

1. **ATT(g,t)**: Group-time specific treatment effects estimated using CSDiD
2. **Group aggregation**: Average effects by treatment cohort
3. **Calendar aggregation**: Average effects by calendar time
4. **Event-time aggregation**: Dynamic effects comparable to event-study

### Files Saved
- `../output/att_gt_results.csv`: All ATT(g,t) estimates
- `../output/att_gt_heatmap.png`: Heatmap of group-time effects
- `../output/csdid_aggregation_group.png`: Group aggregation plot
- `../output/csdid_aggregation_calendar.png`: Calendar aggregation plot
- `../output/csdid_event_study.png`: Event-time aggregation plot
- `../output/csdid_event_time.csv`: Event-time aggregation results
- `../output/comparison_csdid_twfe.csv`: Comparison table
- `../output/comparison_plot_csdid_twfe.png`: Combined comparison plot

In [ ]:
# Final summary
print("="*80)
print("SUMMARY OF QUESTION 2 RESULTS")
print("="*80)

print(f"\n1. ATT(g,t) Estimates: {len(att_gt_results['att'])} group-time effects")

print(f"\n2. Aggregations:")
print(f"   - By Group: {len(agg_group_df)} cohort-specific ATTs")
print(f"   - By Calendar: {len(agg_time_df)} time-specific ATTs")
print(f"   - By Event-Time: {len(agg_event_df)} dynamic effects")

print(f"\n3. Overall ATT: {agg_group['overall.att']:.4f} (SE: {agg_group['overall.se']:.4f})")

print("\n4. Output files generated:")
print("   - att_gt_results.csv")
print("   - att_gt_heatmap.png")
print("   - csdid_aggregation_group.png")
print("   - csdid_aggregation_calendar.png")
print("   - csdid_event_study.png")
print("   - csdid_event_time.csv")
print("   - comparison_csdid_twfe.csv")
print("   - comparison_plot_csdid_twfe.png")